# WaveletSurfaceNet - REGION-COMPOSED training (A100 80GB -> Google Drive)
Trains the **region-composed** model (sharp edges: per-region fields + convex/concave junction composition,
per-piece +5% bound, wavelet edge-refiner) on ModelNet40 with **bf16 + 8-bit Adam**, auto-resuming across
Colab sessions. At the end it renders the 4-panel suite at 128^3 and saves a **zip of renders + weights to
your Drive**. Recipe: mixed base | train 64^3 / eval 128^3 | context+region | seg | dyn-eps + collapse.


## 1 - GPU (expect A100 80GB)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "need a GPU runtime"
print("bf16 supported:", torch.cuda.is_bf16_supported())


## 2 - Configuration


In [ ]:
REPO_URL = "https://github.com/OlegJakushkin/Points_as_supertoroids.git"   # redirects to WaveletSurfaceNet
BRANCH   = "main"
DRIVE_WS = "/content/drive/MyDrive/wsn_composed"   # persistent workspace in Drive
LOCAL    = "/content/wsn"

OUT    = "waveshape"      # checkpoints -> assets/<OUT>.pt (best-by-val) + assets/<OUT>_latest.pt (rolling)
EPOCHS = 16
BATCH  = 12               # A100 80GB, bf16: MEASURED ~2.75 GB/shape retained graph at train-res 64 (B=2*BATCH,
#                           clean+noisy draws) -> BATCH 12 = B 24 ~ 66 GB on the 80 GB card. Do NOT raise past 13.
print(f"config OK | epochs {EPOCHS} | batch {BATCH} (train res 64 is the code default; eval/render at 128)")


## 3 - Mount Drive


In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_WS, exist_ok=True)


## 4 - Get the code


In [ ]:
import os, subprocess
if os.path.isdir(LOCAL):
    subprocess.run(["git", "-C", LOCAL, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", LOCAL, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", LOCAL, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, LOCAL], check=True)
os.chdir(LOCAL); print("repo at", os.getcwd())


## 5 - Persist outputs to Drive (checkpoints/renders/data survive disconnects; auto-resume)


In [ ]:
import os, shutil
for sub in ["assets", "data", "renders"]:
    dst = f"{DRIVE_WS}/{sub}"; os.makedirs(dst, exist_ok=True)
    src = f"{LOCAL}/{sub}"
    if os.path.islink(src): continue
    if os.path.isdir(src):                       # seed Drive with repo assets (teapot/bunny/chair inputs)
        for f in os.listdir(src):
            if not os.path.exists(f"{dst}/{f}"): shutil.move(f"{src}/{f}", f"{dst}/{f}")
        shutil.rmtree(src)
    os.symlink(dst, src)
print("assets/, data/, renders/ -> Drive")


## 6 - Dependencies (bitsandbytes gives the 8-bit Adam; train.py falls back gracefully if it fails)


In [ ]:
!pip install -q trimesh rtree scikit-image networkx scipy matplotlib bitsandbytes
import trimesh, torch
print("trimesh", trimesh.__version__, "| torch", torch.__version__)
try:
    import bitsandbytes as bnb; print("bitsandbytes", bnb.__version__, "-> Adam8bit ACTIVE")
except Exception as e:
    print("bitsandbytes unavailable (", e, ") -> train.py falls back to fp32-state Adam")


## 7 - Build the training data (ModelNet40 -> cloud cache; cached in Drive, one time ~30 min)


In [ ]:
import os, glob, zipfile, urllib.request
import numpy as np, torch
from waveshape import eval3d as E
from waveshape.datasets import load_mesh_normalized

DENSE, CAP = 1536, 9843
if not os.path.isdir("data/ModelNet40"):
    z = "data/ModelNet40.zip"
    if not os.path.exists(z):
        print("downloading ModelNet40 (~2GB, one time)...", flush=True)
        urllib.request.urlretrieve("http://modelnet.cs.princeton.edu/ModelNet40.zip", z)
    print("extracting...", flush=True)
    with zipfile.ZipFile(z) as zf: zf.extractall("data")

if not os.path.exists("data/se_clouds.pt"):
    files = sorted(glob.glob("data/ModelNet40/*/train/*.off"))[:CAP]
    print(f"building cloud cache from {len(files)} meshes...", flush=True)
    Ps, Ns = [], []
    for i, p in enumerate(files):
        try:
            m = load_mesh_normalized(p, max_faces=200000)
            Pp, Nn = E.sample_cloud(m, n=DENSE, noise=0.0, seed=0)
            if np.isfinite(Pp).all() and np.isfinite(Nn).all():
                Ps.append(Pp.astype(np.float32)); Ns.append(Nn.astype(np.float32))
        except Exception:
            continue
        if (i + 1) % 1000 == 0: print(f"  {i+1}/{len(files)} ({len(Ps)} ok)", flush=True)
    torch.save({"P": torch.tensor(np.stack(Ps)), "N": torch.tensor(np.stack(Ns))}, "data/se_clouds.pt")
    print(f"cached {len(Ps)} clouds", flush=True)
else:
    print("data/se_clouds.pt already cached in Drive", flush=True)


## 8 - Train (bf16 + Adam8bit; region pool auto-built and cached; re-run this cell to RESUME)
First run also builds the per-shape region cache (`assets/region_pool_<M>.pt`, ~5 min, then free forever).
The RECIPE line printed at start confirms: REGION-COMPOSED anchor+target | bf16 | Adam8bit.


In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"
latest = f"assets/{OUT}_latest.pt"
args = f"--out {OUT} --batch {BATCH} --epochs {EPOCHS}"
if os.path.exists(latest):                       # resume across sessions (continues at the stored epoch)
    args += f" --resume {latest}"
print("python train.py", args, flush=True)
!python train.py {args} 2>&1 | tee -a {DRIVE_WS}/train_{OUT}.log


## 9 - Render the 4-panel suite at 128^3 (favourites / thin / noise sweep; scenes auto-skips without scans)


In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"; os.environ["CKPT"] = f"assets/{OUT}.pt"; os.environ["EVAL_RES"] = "128"
!python render_suite.py
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("renders/suite_*.png")): display(Image(p, width=1000))


## 10 - Zip renders + weights -> Drive


In [ ]:
import glob, os, time, zipfile
ts = time.strftime("%Y%m%d_%H%M")
zpath = f"/content/drive/MyDrive/wsn_composed_{ts}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in [f"assets/{OUT}.pt", f"assets/{OUT}_latest.pt", "renders/suite_*.png",
                "renders/wsn_train_hist.json"]:
        for f in glob.glob(pat):
            z.write(f, arcname=os.path.basename(f)); print("  +", f)
print("
saved:", zpath, f"({os.path.getsize(zpath)/1e6:.1f} MB)")
